In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1tMeLPtVrclN_VhbwgNJ7TznyewJkzpbZ", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_01_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Notebook 1: Crafting Effective Tool Descriptions

**Claude Certified Architect Prep** | Pod 2: Tool Design & MCP Integration

---

In this notebook, you will learn to write tool descriptions that Claude can reliably distinguish, select, and use correctly. Poor tool descriptions are the **#1 cause of tool misrouting** in production agents.

By the end of this notebook you will be able to:
- Identify why vague descriptions cause tool selection failures
- Apply a 5-component anatomy for writing precise descriptions
- Rewrite, rename, and split overlapping tools
- Detect system prompt keyword traps
- Build an automated linter for tool description quality

**No API key required** -- all exercises use mock Claude responses so you can focus on the design patterns.

---

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/tool-design-mcp/practice/1/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
# ============================================================
# Setup: Install dependencies
# ============================================================
# Run this cell first. All packages are available on Colab by default,
# but we pin versions for reproducibility.

!pip install -q matplotlib numpy

import json
import re
import textwrap
from typing import Any
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Utility: pretty-print JSON
def pp(obj: Any) -> None:
    """Pretty-print a Python object as formatted JSON."""
    print(json.dumps(obj, indent=2, default=str))

print("Setup complete.")

In [ ]:
#@title 🎧 Listen: Warmup Problem
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_warmup_problem.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 2: Warm-Up -- The Problem

Imagine you are building a **Customer Support Agent** that uses Claude with three tools. A customer writes:

> *"I need to update my shipping address for order #4821."*

Claude must pick the correct tool. But what happens when your descriptions are vague?

Let's simulate this.

In [ ]:
#@title 🎧 Listen: Bad Tools Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_bad_tools_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ============================================================
# 2A: Define poorly-described tools
# ============================================================

bad_tools = [
    {
        "name": "handle_order",
        "description": "Handles orders.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "action": {"type": "string"}
            }
        }
    },
    {
        "name": "update_info",
        "description": "Updates information in the system.",
        "input_schema": {
            "type": "object",
            "properties": {
                "field": {"type": "string"},
                "value": {"type": "string"}
            }
        }
    },
    {
        "name": "customer_service",
        "description": "Provides customer service.",
        "input_schema": {
            "type": "object",
            "properties": {
                "request": {"type": "string"}
            }
        }
    },
]

print("Bad Tool Definitions:")
print("=" * 60)
for tool in bad_tools:
    print(f"\n  Tool: {tool['name']}")
    print(f"  Desc: {tool['description']}")
    print(f"  Params: {list(tool['input_schema']['properties'].keys())}")

In [ ]:
# ============================================================
# 2B: Simulate Claude picking the wrong tool
# ============================================================

# This mock simulates what actually happens: Claude sees three tools that
# all *could* handle the request and picks based on surface-level keyword
# matching. With vague descriptions, the selection is essentially random.

def mock_claude_tool_selection(user_message: str, tools: list[dict]) -> dict:
    """
    Simulates Claude's tool selection logic.
    With vague descriptions, the model falls back to keyword overlap.
    """
    scores = {}
    user_words = set(user_message.lower().split())

    for tool in tools:
        desc_words = set(tool["description"].lower().split())
        name_words = set(tool["name"].lower().replace("_", " ").split())
        all_tool_words = desc_words | name_words

        # Simple keyword overlap score (mimics how vague descriptions
        # force the model to rely on shallow matching)
        overlap = user_words & all_tool_words
        scores[tool["name"]] = len(overlap)

    selected = max(scores, key=scores.get)
    return {
        "selected_tool": selected,
        "scores": scores,
        "confidence": "LOW -- descriptions are too vague to differentiate"
    }

# Customer request
user_msg = "I need to update my shipping address for order #4821."

result = mock_claude_tool_selection(user_msg, bad_tools)

print(f"Customer: \"{user_msg}\"")
print(f"\nTool Selection Scores:")
for name, score in result["scores"].items():
    marker = " <-- SELECTED" if name == result["selected_tool"] else ""
    print(f"  {name}: {score} keyword overlaps{marker}")
print(f"\nConfidence: {result['confidence']}")
print(f"\nProblem: Claude selected '{result['selected_tool']}' but ANY of these")
print("tools could be correct because none of the descriptions explain")
print("their actual purpose or boundaries.")

In [ ]:
# ============================================================
# 2C: Show the cascade of failures
# ============================================================

# When tools overlap, different phrasings of the SAME request
# route to different tools. This is a reliability disaster.

test_messages = [
    "I need to update my shipping address for order #4821.",
    "Change the address on my order #4821 please.",
    "My order #4821 has the wrong address, can you fix it?",
    "Order #4821 -- the shipping destination needs to be corrected.",
    "Please help me with my order #4821 address issue.",
]

print("Same intent, different phrasings -> different tool selections:")
print("=" * 65)
for msg in test_messages:
    result = mock_claude_tool_selection(msg, bad_tools)
    print(f"\n  \"{msg[:55]}...\"")
    print(f"  -> {result['selected_tool']}")

unique_selections = set(
    mock_claude_tool_selection(m, bad_tools)["selected_tool"]
    for m in test_messages
)
print(f"\nTools selected across 5 equivalent requests: {len(unique_selections)}")
print(f"Unique tools used: {unique_selections}")
print("\nVerdict: INCONSISTENT. The same customer intent routes to")
print("different tools depending on word choice. This is broken.")

### What went wrong?

| Problem | Example |
|---|---|
| **No specificity** | "Handles orders" -- does it create? cancel? modify? track? |
| **No input guidance** | What format is `action`? What values are valid? |
| **No output description** | What does each tool return? |
| **No use-case boundaries** | When should I pick `handle_order` vs `customer_service`? |
| **Overlapping scope** | All three tools could plausibly "update" something |

The root cause: **the model has no signal to differentiate tools**. Let's fix this.

---

In [ ]:
#@title 🎧 Listen: Anatomy Explanation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_anatomy_explanation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 3: Core Concept -- Tool Description Anatomy

Every great tool description has **5 components**:

1. **What it does** -- A precise one-sentence summary of the action
2. **Expected inputs** -- Format, constraints, and valid values for each parameter
3. **What it returns** -- The structure and content of the response
4. **When to use it** -- Positive routing signals (use this tool WHEN...)
5. **When NOT to use it** -- Negative routing signals (do NOT use this tool for...)

Component #5 is the secret weapon. Most developers skip it, and that is exactly where misrouting happens.

In [ ]:
#@title 🎧 Listen: Rewrite Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_rewrite_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ============================================================
# 3A: Before vs After -- handle_order tool
# ============================================================

before_description = "Handles orders."

after_description = (
    "Modifies the status of an existing customer order. "
    "Use this to cancel an order, change its priority, or put it on hold. "
    "Requires the order_id (format: numeric, e.g. '4821') and an action "
    "which must be one of: 'cancel', 'hold', 'prioritize'. "
    "Returns the updated order object with new status and timestamp. "
    "Do NOT use this tool to update shipping addresses or payment info -- "
    "use update_shipping_address or update_payment_method instead."
)

print("BEFORE:")
print(f"  \"{before_description}\"")
print(f"  Word count: {len(before_description.split())}")
print()
print("AFTER:")
print(f"  \"{after_description}\"")
print(f"  Word count: {len(after_description.split())}")

# Annotate the 5 components
print("\n\n5-Component Breakdown:")
print("=" * 60)
components = {
    "1. What it does": "Modifies the status of an existing customer order.",
    "2. Expected inputs": "order_id (numeric, e.g. '4821'), action ('cancel'|'hold'|'prioritize')",
    "3. What it returns": "Updated order object with new status and timestamp.",
    "4. When to use": "Cancel, change priority, or hold an order.",
    "5. When NOT to use": "NOT for address updates or payment changes."
}
for component, text in components.items():
    print(f"\n  {component}")
    print(f"    \"{text}\"")

In [ ]:
# ============================================================
# 3B: Full rewrite of all three tools
# ============================================================

good_tools = [
    {
        "name": "modify_order_status",
        "description": (
            "Changes the status of an existing customer order. "
            "Use this to cancel an order, change its shipping priority, "
            "or place it on hold. Requires the order_id (numeric string, "
            "e.g. '4821') and a status_action which must be one of: "
            "'cancel', 'hold', 'expedite'. Returns the updated order "
            "object with the new status, previous status, and timestamp. "
            "Do NOT use for address changes (use update_shipping_address), "
            "payment changes (use update_payment_method), or order tracking "
            "(use get_order_tracking)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "Numeric order ID, e.g. '4821'"
                },
                "status_action": {
                    "type": "string",
                    "enum": ["cancel", "hold", "expedite"],
                    "description": "The status change to apply"
                }
            },
            "required": ["order_id", "status_action"]
        }
    },
    {
        "name": "update_shipping_address",
        "description": (
            "Updates the shipping destination for an existing order that "
            "has not yet shipped. Requires the order_id (numeric string) "
            "and the new address as a structured object with street, city, "
            "state, and zip fields. Returns confirmation with the old and "
            "new addresses. Fails if the order has already shipped -- in "
            "that case, direct the customer to contact the carrier. "
            "Do NOT use for billing address changes (use update_billing_info) "
            "or order cancellations (use modify_order_status)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "Numeric order ID, e.g. '4821'"
                },
                "new_address": {
                    "type": "object",
                    "properties": {
                        "street": {"type": "string"},
                        "city": {"type": "string"},
                        "state": {"type": "string"},
                        "zip": {"type": "string"}
                    },
                    "required": ["street", "city", "state", "zip"]
                }
            },
            "required": ["order_id", "new_address"]
        }
    },
    {
        "name": "get_order_tracking",
        "description": (
            "Retrieves the current shipping status and tracking information "
            "for an order. Returns carrier name, tracking number, estimated "
            "delivery date, and a list of status events (e.g., 'picked up', "
            "'in transit', 'out for delivery'). Requires only the order_id "
            "(numeric string). Use when the customer asks WHERE their order "
            "is or WHEN it will arrive. Do NOT use to change anything about "
            "the order -- this is read-only. For modifications, use "
            "modify_order_status or update_shipping_address."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "Numeric order ID, e.g. '4821'"
                }
            },
            "required": ["order_id"]
        }
    },
]

print("Improved Tool Definitions:")
print("=" * 60)
for tool in good_tools:
    print(f"\n  Tool: {tool['name']}")
    # Wrap description for readability
    wrapped = textwrap.fill(tool["description"], width=58, initial_indent="  Desc: ", subsequent_indent="        ")
    print(wrapped)
    params = tool["input_schema"]["properties"]
    print(f"  Params: {list(params.keys())}")
    if "required" in tool["input_schema"]:
        print(f"  Required: {tool['input_schema']['required']}")

In [ ]:
# ============================================================
# 3C: Re-run the routing test with improved tools
# ============================================================

def mock_claude_improved_selection(user_message: str, tools: list[dict]) -> dict:
    """
    Simulates Claude's tool selection with well-described tools.
    Better descriptions give the model strong signal for routing.
    """
    scores = {}
    msg_lower = user_message.lower()

    for tool in tools:
        desc = tool["description"].lower()
        name = tool["name"].lower().replace("_", " ")
        score = 0

        # Semantic matching: check for key phrases in description
        # that align with the user request
        routing_signals = {
            "address": 3,
            "shipping": 2,
            "destination": 3,
            "cancel": 3,
            "hold": 3,
            "expedite": 3,
            "priority": 2,
            "tracking": 3,
            "where": 2,
            "when will": 2,
            "status": 1,
            "update": 1,
            "change": 1,
            "fix": 1,
        }

        for keyword, weight in routing_signals.items():
            if keyword in msg_lower and keyword in desc:
                score += weight
            # Negative routing: penalize if desc says NOT to use for this
            not_pattern = f"do not use.*{keyword}"
            if keyword in msg_lower and re.search(not_pattern, desc):
                score -= weight * 2  # Strong penalty for negative match

        scores[tool["name"]] = score

    selected = max(scores, key=scores.get)
    max_score = max(scores.values())
    second_score = sorted(scores.values(), reverse=True)[1]
    gap = max_score - second_score

    confidence = "HIGH" if gap >= 3 else "MEDIUM" if gap >= 1 else "LOW"

    return {
        "selected_tool": selected,
        "scores": scores,
        "confidence": confidence
    }

print("Re-testing with IMPROVED tool descriptions:")
print("=" * 65)
for msg in test_messages:
    result = mock_claude_improved_selection(msg, good_tools)
    print(f"\n  \"{msg[:55]}...\"")
    print(f"  -> {result['selected_tool']} (confidence: {result['confidence']})")

unique_selections = set(
    mock_claude_improved_selection(m, good_tools)["selected_tool"]
    for m in test_messages
)
print(f"\nTools selected across 5 equivalent requests: {len(unique_selections)}")
print(f"Unique tools used: {unique_selections}")
print("\nVerdict: CONSISTENT. All phrasings route to update_shipping_address.")

### Key Takeaway

The **same underlying model** with **better descriptions** produces **reliable routing**. You did not need a more powerful model -- you needed clearer tool definitions.

This is the single most impactful change you can make to any tool-use agent.

---

In [ ]:
#@title 🎧 Listen: Exercise Rewrite
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_exercise_rewrite.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 4: Guided Exercise -- Rewrite Descriptions

Below are 3 poorly-written tool descriptions from a **Financial Advisor Agent**. Your job: rewrite each one using the 5-component anatomy.

**Scoring rubric** (we will automate this in Section 5):
- Specificity: Does it say exactly what the tool does?
- Boundary clarity: Does it say when NOT to use it?
- Input format: Are parameter formats/constraints documented?
- Output format: Is the return value described?
- Use-case examples: Are there concrete routing signals?

In [ ]:
# ============================================================
# 4A: Exercise -- Three tools to rewrite
# ============================================================

poor_tools = [
    {
        "name": "get_data",
        "description": "Gets financial data.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            }
        }
    },
    {
        "name": "analyze",
        "description": "Analyzes things.",
        "input_schema": {
            "type": "object",
            "properties": {
                "data": {"type": "string"},
                "type": {"type": "string"}
            }
        }
    },
    {
        "name": "send_report",
        "description": "Sends a report.",
        "input_schema": {
            "type": "object",
            "properties": {
                "report": {"type": "string"},
                "recipient": {"type": "string"}
            }
        }
    },
]

print("YOUR TASK: Rewrite each description using the 5-component anatomy.")
print("=" * 65)
for i, tool in enumerate(poor_tools, 1):
    print(f"\n  Tool #{i}: {tool['name']}")
    print(f"  Current description: \"{tool['description']}\"")
    print(f"  Parameters: {list(tool['input_schema']['properties'].keys())}")
print()
print("Components to include in each rewrite:")
print("  1. What it does (precise action)")
print("  2. Expected inputs (format, constraints, valid values)")
print("  3. What it returns (structure and content)")
print("  4. When to use it (positive routing signals)")
print("  5. When NOT to use it (negative routing signals)")

In [ ]:
# ============================================================
# 4B: YOUR REWRITES -- Fill in below
# ============================================================

# TODO: Rewrite each tool description. Replace the placeholder strings.

my_rewrite_1 = {
    "name": "get_data",  # TODO: Consider renaming this tool too
    "description": (
        # TODO: Write your improved description here.
        # Remember: what it does, inputs, outputs, when to use, when NOT to use.
        "Gets financial data."  # <-- Replace this!
    ),
}

my_rewrite_2 = {
    "name": "analyze",  # TODO: Consider renaming this tool too
    "description": (
        # TODO: Write your improved description here.
        "Analyzes things."  # <-- Replace this!
    ),
}

my_rewrite_3 = {
    "name": "send_report",  # TODO: Consider renaming this tool too
    "description": (
        # TODO: Write your improved description here.
        "Sends a report."  # <-- Replace this!
    ),
}

# Print your rewrites
print("Your Rewrites:")
print("=" * 60)
for i, tool in enumerate([my_rewrite_1, my_rewrite_2, my_rewrite_3], 1):
    print(f"\n  Tool #{i}: {tool['name']}")
    wrapped = textwrap.fill(tool["description"], width=55,
                            initial_indent="  Desc: ",
                            subsequent_indent="        ")
    print(wrapped)

In [ ]:
# ============================================================
# 4C: SOLUTION -- Reference rewrites
# ============================================================

# Expand this cell after attempting your own rewrites.

solution_tools = [
    {
        "name": "fetch_stock_quote",
        "description": (
            "Retrieves the current or historical stock price for a single "
            "ticker symbol. Requires 'ticker' (uppercase string, e.g. 'AAPL', "
            "'MSFT') and an optional 'date' (ISO format 'YYYY-MM-DD'; omit "
            "for the latest quote). Returns a JSON object with: ticker, price, "
            "currency, timestamp, daily_change_pct, and volume. Use this when "
            "the user asks for the price of a specific stock or wants to compare "
            "prices on specific dates. Do NOT use for portfolio-level analysis "
            "(use analyze_portfolio_risk), mutual fund NAVs (use fetch_fund_nav), "
            "or cryptocurrency prices (use fetch_crypto_price)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Uppercase stock ticker, e.g. 'AAPL'"
                },
                "date": {
                    "type": "string",
                    "description": "Optional ISO date 'YYYY-MM-DD' for historical price"
                }
            },
            "required": ["ticker"]
        }
    },
    {
        "name": "analyze_portfolio_risk",
        "description": (
            "Calculates risk metrics for a portfolio of holdings. Requires "
            "'holdings' as a list of objects, each with 'ticker' (string) "
            "and 'weight' (float, 0-1, must sum to 1.0). Optionally accepts "
            "'period' ('1y', '3y', '5y'; default '1y') for the lookback "
            "window. Returns a JSON object with: sharpe_ratio, max_drawdown, "
            "volatility, beta, var_95 (Value at Risk at 95% confidence), and "
            "a per-holding risk contribution breakdown. Use when the user asks "
            "about portfolio risk, diversification, or wants to evaluate their "
            "asset allocation. Do NOT use for individual stock prices (use "
            "fetch_stock_quote) or generating PDF reports (use "
            "generate_portfolio_report)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "holdings": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "ticker": {"type": "string"},
                            "weight": {"type": "number"}
                        }
                    }
                },
                "period": {
                    "type": "string",
                    "enum": ["1y", "3y", "5y"],
                    "description": "Lookback period, default '1y'"
                }
            },
            "required": ["holdings"]
        }
    },
    {
        "name": "email_portfolio_report",
        "description": (
            "Generates a formatted PDF portfolio report and emails it to the "
            "specified recipient. Requires 'report_type' (one of: 'monthly_summary', "
            "'risk_analysis', 'tax_lots') and 'recipient_email' (valid email "
            "address string). Optionally accepts 'date_range' with 'start' and "
            "'end' in ISO format. Returns a confirmation object with: email_status "
            "('sent'|'failed'), report_url (temporary download link valid for "
            "24 hours), and page_count. Use when the user explicitly asks to "
            "receive a report via email or wants a downloadable PDF. Do NOT use "
            "for real-time data queries (use fetch_stock_quote) or interactive "
            "risk analysis (use analyze_portfolio_risk)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "report_type": {
                    "type": "string",
                    "enum": ["monthly_summary", "risk_analysis", "tax_lots"]
                },
                "recipient_email": {
                    "type": "string",
                    "description": "Valid email address for delivery"
                },
                "date_range": {
                    "type": "object",
                    "properties": {
                        "start": {"type": "string"},
                        "end": {"type": "string"}
                    }
                }
            },
            "required": ["report_type", "recipient_email"]
        }
    },
]

print("SOLUTION -- Reference Rewrites:")
print("=" * 65)
for i, tool in enumerate(solution_tools, 1):
    print(f"\nTool #{i}: {tool['name']}")
    print("-" * 40)
    wrapped = textwrap.fill(tool["description"], width=60,
                            initial_indent="  ",
                            subsequent_indent="  ")
    print(wrapped)
    print()
    # Highlight the 5 components
    desc = tool["description"]
    print(f"  Component checklist:")
    print(f"    [{'x' if 'Retrieves' in desc or 'Calculates' in desc or 'Generates' in desc else ' '}] What it does")
    print(f"    [{'x' if 'Requires' in desc else ' '}] Expected inputs")
    print(f"    [{'x' if 'Returns' in desc else ' '}] What it returns")
    print(f"    [{'x' if 'Use' in desc and 'when' in desc.lower() else ' '}] When to use")
    print(f"    [{'x' if 'Do NOT' in desc else ' '}] When NOT to use")

### What changed?

| Before | After |
|---|---|
| `get_data` | `fetch_stock_quote` -- name alone tells you the scope |
| `analyze` | `analyze_portfolio_risk` -- specifies what kind of analysis |
| `send_report` | `email_portfolio_report` -- clarifies delivery mechanism |
| No input guidance | Explicit formats, enums, and required fields |
| No boundary | Every tool says what it is NOT for |

---

In [ ]:
#@title 🎧 Listen: Scorer Explanation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_scorer_explanation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 5: Visual -- Description Quality Scorer

Let's build an automated scorer that evaluates tool descriptions on the 5 quality criteria. This makes the abstract rules concrete and measurable.

In [ ]:
# ============================================================
# 5A: Build the Description Quality Scorer
# ============================================================

def score_tool_description(tool: dict) -> dict:
    """
    Scores a tool description on 5 criteria, each 0-10.
    Returns a dict with individual scores and an overall grade.
    """
    desc = tool.get("description", "")
    name = tool.get("name", "")
    schema = tool.get("input_schema", {})
    props = schema.get("properties", {})
    desc_lower = desc.lower()
    word_count = len(desc.split())

    scores = {}

    # 1. SPECIFICITY (0-10)
    # Does it say exactly what the tool does?
    specificity = 0
    action_verbs = [
        "retrieves", "fetches", "creates", "updates", "deletes",
        "calculates", "generates", "sends", "searches", "modifies",
        "validates", "converts", "exports", "imports", "analyzes"
    ]
    if any(v in desc_lower for v in action_verbs):
        specificity += 3  # Starts with a clear action verb
    if word_count >= 20:
        specificity += 2  # Sufficient detail
    if word_count >= 40:
        specificity += 2  # Rich detail
    # Penalize ultra-vague descriptions
    vague_phrases = ["handles", "does stuff", "manages things", "works with"]
    if any(v in desc_lower for v in vague_phrases):
        specificity -= 2
    # Check that the tool name is descriptive (not generic)
    generic_names = ["handle", "process", "do", "run", "execute", "manage", "get_data"]
    if name.lower() in generic_names:
        specificity -= 1
    if len(name) > 10:
        specificity += 1  # Longer names tend to be more specific
    # Check for concrete examples
    if "e.g." in desc_lower or "for example" in desc_lower:
        specificity += 2
    scores["specificity"] = max(0, min(10, specificity))

    # 2. BOUNDARY CLARITY (0-10)
    # Does it say when NOT to use it?
    boundary = 0
    if "do not use" in desc_lower or "don't use" in desc_lower:
        boundary += 4
    if "instead" in desc_lower or "use " in desc_lower:
        boundary += 2  # References alternative tools
    # Count how many alternative tools are mentioned by name
    alt_tool_refs = re.findall(r"use (\w+_\w+)", desc_lower)
    boundary += min(len(alt_tool_refs), 4)  # Up to 4 bonus points
    scores["boundary_clarity"] = max(0, min(10, boundary))

    # 3. INPUT FORMAT (0-10)
    # Are parameter formats/constraints documented?
    input_score = 0
    # Check if properties have descriptions
    props_with_desc = sum(1 for p in props.values()
                         if isinstance(p, dict) and "description" in p)
    if len(props) > 0:
        input_score += int((props_with_desc / len(props)) * 4)
    # Check for enums
    props_with_enum = sum(1 for p in props.values()
                          if isinstance(p, dict) and "enum" in p)
    input_score += min(props_with_enum * 2, 4)
    # Check for required fields
    if "required" in schema:
        input_score += 2
    # Check if description mentions input formats
    if "requires" in desc_lower or "accepts" in desc_lower:
        input_score += 1
    scores["input_format"] = max(0, min(10, input_score))

    # 4. OUTPUT FORMAT (0-10)
    # Is the return value described?
    output_score = 0
    if "returns" in desc_lower:
        output_score += 4
    # Check for specific return field mentions
    return_match = re.search(r"returns.*?(?:\.|$)", desc_lower)
    if return_match:
        return_text = return_match.group()
        # Count specific field names mentioned
        fields_mentioned = len(re.findall(r"\w+_\w+", return_text))
        output_score += min(fields_mentioned, 4)
    if "json" in desc_lower or "object" in desc_lower:
        output_score += 2
    scores["output_format"] = max(0, min(10, output_score))

    # 5. USE-CASE SIGNALS (0-10)
    # Are there concrete routing signals?
    usecase = 0
    if "use this when" in desc_lower or "use when" in desc_lower:
        usecase += 3
    if "asks for" in desc_lower or "asks about" in desc_lower or "wants to" in desc_lower:
        usecase += 3  # Describes user intent
    # Check for quoted examples or concrete scenarios
    if '"' in desc or "'" in desc:
        usecase += 2
    if "e.g." in desc_lower or "for example" in desc_lower or "such as" in desc_lower:
        usecase += 2
    scores["usecase_signals"] = max(0, min(10, usecase))

    # Overall
    total = sum(scores.values())
    max_total = len(scores) * 10
    pct = (total / max_total) * 100

    if pct >= 80:
        grade = "A"
    elif pct >= 60:
        grade = "B"
    elif pct >= 40:
        grade = "C"
    elif pct >= 20:
        grade = "D"
    else:
        grade = "F"

    return {
        "tool_name": name,
        "scores": scores,
        "total": total,
        "max_total": max_total,
        "percentage": round(pct, 1),
        "grade": grade
    }

# Test on our bad and good tools
print("Scoring BAD tools:")
print("=" * 50)
for tool in bad_tools:
    result = score_tool_description(tool)
    print(f"\n  {result['tool_name']}: {result['total']}/{result['max_total']} ({result['grade']})")
    for criterion, score in result["scores"].items():
        bar = "#" * score + "." * (10 - score)
        print(f"    {criterion:20s} [{bar}] {score}/10")

print("\n\nScoring GOOD tools:")
print("=" * 50)
for tool in good_tools:
    result = score_tool_description(tool)
    print(f"\n  {result['tool_name']}: {result['total']}/{result['max_total']} ({result['grade']})")
    for criterion, score in result["scores"].items():
        bar = "#" * score + "." * (10 - score)
        print(f"    {criterion:20s} [{bar}] {score}/10")

In [ ]:
#@title 🎧 Listen: Radar Chart Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_radar_chart_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ============================================================
# 5B: Visualization -- Radar chart comparison
# ============================================================

def plot_radar_comparison(bad_tools: list[dict], good_tools: list[dict]) -> None:
    """
    Creates a radar chart comparing bad vs good tool description scores.
    """
    categories = ["Specificity", "Boundary\nClarity", "Input\nFormat",
                   "Output\nFormat", "Use-Case\nSignals"]

    # Average scores across tools
    bad_scores_all = [score_tool_description(t)["scores"] for t in bad_tools]
    good_scores_all = [score_tool_description(t)["scores"] for t in good_tools]

    keys = ["specificity", "boundary_clarity", "input_format",
            "output_format", "usecase_signals"]

    bad_avg = [np.mean([s[k] for s in bad_scores_all]) for k in keys]
    good_avg = [np.mean([s[k] for s in good_scores_all]) for k in keys]

    # Number of variables
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]  # Close the polygon

    bad_avg += bad_avg[:1]
    good_avg += good_avg[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    # Draw the bad tools
    ax.plot(angles, bad_avg, 'o-', linewidth=2, color='#ff6b6b', label='Before (vague)')
    ax.fill(angles, bad_avg, alpha=0.15, color='#ff6b6b')

    # Draw the good tools
    ax.plot(angles, good_avg, 'o-', linewidth=2, color='#51cf66', label='After (precise)')
    ax.fill(angles, good_avg, alpha=0.15, color='#51cf66')

    # Labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=12, fontweight='bold')
    ax.set_ylim(0, 10)
    ax.set_yticks([2, 4, 6, 8, 10])
    ax.set_yticklabels(['2', '4', '6', '8', '10'], size=9, color='gray')
    ax.set_title("Tool Description Quality: Before vs After",
                 size=16, fontweight='bold', pad=30)
    ax.legend(loc='lower right', fontsize=12, bbox_to_anchor=(1.15, -0.05))

    plt.tight_layout()
    plt.show()

plot_radar_comparison(bad_tools, good_tools)

In [ ]:
# ============================================================
# 5C: Bar chart -- Individual tool scores
# ============================================================

def plot_individual_scores(tools: list[dict], title: str) -> None:
    """Bar chart showing each tool's score breakdown."""
    categories = ["Specificity", "Boundary", "Input Fmt", "Output Fmt", "Use-Case"]
    keys = ["specificity", "boundary_clarity", "input_format",
            "output_format", "usecase_signals"]

    fig, ax = plt.subplots(figsize=(12, 5))

    tool_names = [t["name"] for t in tools]
    x = np.arange(len(tool_names))
    width = 0.15

    colors = ['#4c6ef5', '#ae3ec9', '#f59f00', '#20c997', '#ff6b6b']

    for i, (key, cat, color) in enumerate(zip(keys, categories, colors)):
        scores_list = [score_tool_description(t)["scores"][key] for t in tools]
        ax.bar(x + i * width, scores_list, width, label=cat, color=color, alpha=0.85)

    ax.set_xlabel('Tool', fontsize=12)
    ax.set_ylabel('Score (0-10)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(tool_names, fontsize=10)
    ax.set_ylim(0, 11)
    ax.legend(fontsize=9, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.12))

    plt.tight_layout()
    plt.show()

plot_individual_scores(bad_tools, "Score Breakdown: POOR Tool Descriptions")
plot_individual_scores(good_tools, "Score Breakdown: IMPROVED Tool Descriptions")

### Reading the Charts

- **Before (red)**: Every criterion is near zero. The model has almost no signal to work with.
- **After (green)**: Scores jump across the board. The biggest gains come from boundary clarity and use-case signals -- exactly the components most developers skip.

The quality scorer gives you a **concrete, repeatable way** to audit tool descriptions before deploying an agent.

---

In [ ]:
#@title 🎧 Listen: Split Exercise
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_split_exercise.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 6: Exercise -- Rename and Split Overlapping Tools

This is a common real-world problem. You have two tools with overlapping names:

- `analyze_content` -- "Analyzes content and returns insights."
- `analyze_document` -- "Analyzes a document and provides analysis."

When a user says *"Analyze this PDF report for key findings"*, which tool does Claude pick? **It cannot reliably tell.** The names overlap, the descriptions overlap, and neither specifies boundaries.

**Your task**: Split these into **3 purpose-specific tools** with clear, non-overlapping descriptions.

Think about what "analyze" actually means in different contexts:
- Summarizing key points?
- Extracting structured data (tables, entities)?
- Checking for compliance/policy issues?

In [ ]:
# ============================================================
# 6A: The overlapping tools
# ============================================================

overlapping_tools = [
    {
        "name": "analyze_content",
        "description": "Analyzes content and returns insights.",
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {"type": "string"},
                "analysis_type": {"type": "string"}
            }
        }
    },
    {
        "name": "analyze_document",
        "description": "Analyzes a document and provides analysis.",
        "input_schema": {
            "type": "object",
            "properties": {
                "document_id": {"type": "string"},
                "options": {"type": "string"}
            }
        }
    },
]

# Show the problem
print("OVERLAP ANALYSIS:")
print("=" * 60)
for tool in overlapping_tools:
    result = score_tool_description(tool)
    print(f"\n  {tool['name']}: Grade {result['grade']} ({result['percentage']}%)")

# Show keyword overlap
words_1 = set(overlapping_tools[0]["description"].lower().split())
words_2 = set(overlapping_tools[1]["description"].lower().split())
overlap = words_1 & words_2
print(f"\nShared words in descriptions: {overlap}")
print(f"Overlap ratio: {len(overlap) / len(words_1 | words_2):.0%}")
print("\nVerdict: These tools are nearly indistinguishable to the model.")

In [ ]:
# ============================================================
# 6B: TODO -- Your split into 3 purpose-specific tools
# ============================================================

# TODO: Replace the placeholder descriptions below.
# Each tool should have a unique, well-defined purpose.
# Use the 5-component anatomy for each description.

split_tool_1 = {
    # TODO: Choose a descriptive name
    "name": "summarize_document",  # TODO: Improve this name if needed
    "description": (
        # TODO: Write a complete description.
        # This tool should focus on SUMMARIZING: extracting key points,
        # generating executive summaries, identifying main arguments.
        "YOUR DESCRIPTION HERE"
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            # TODO: Define appropriate parameters
            "document_id": {
                "type": "string",
                "description": "TODO: Add description"
            },
            # TODO: Add more parameters as needed
        },
        "required": ["document_id"]
    }
}

split_tool_2 = {
    # TODO: Choose a descriptive name
    "name": "extract_document_data",  # TODO: Improve this name if needed
    "description": (
        # TODO: Write a complete description.
        # This tool should focus on EXTRACTING structured data: tables,
        # named entities, dates, monetary values, key-value pairs.
        "YOUR DESCRIPTION HERE"
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            # TODO: Define appropriate parameters
            "document_id": {
                "type": "string",
                "description": "TODO: Add description"
            },
            # TODO: Add more parameters as needed
        },
        "required": ["document_id"]
    }
}

split_tool_3 = {
    # TODO: Choose a descriptive name
    "name": "check_document_compliance",  # TODO: Improve this name if needed
    "description": (
        # TODO: Write a complete description.
        # This tool should focus on COMPLIANCE: checking against policies,
        # flagging regulatory issues, verifying required sections.
        "YOUR DESCRIPTION HERE"
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            # TODO: Define appropriate parameters
            "document_id": {
                "type": "string",
                "description": "TODO: Add description"
            },
            # TODO: Add more parameters as needed
        },
        "required": ["document_id"]
    }
}

# Print your split
print("YOUR SPLIT TOOLS:")
print("=" * 60)
for tool in [split_tool_1, split_tool_2, split_tool_3]:
    result = score_tool_description(tool)
    print(f"\n  {tool['name']}: Grade {result['grade']} ({result['percentage']}%)")
    wrapped = textwrap.fill(tool['description'], width=55,
                            initial_indent="  Desc: ",
                            subsequent_indent="        ")
    print(wrapped)

In [ ]:
# ============================================================
# 6C: SOLUTION -- Three purpose-specific tools
# ============================================================

solution_split = [
    {
        "name": "summarize_document_key_points",
        "description": (
            "Generates a structured summary of a document's main arguments, "
            "conclusions, and key points. Requires 'document_id' (string, the "
            "unique identifier from the document store) and 'summary_length' "
            "(one of: 'brief' for 3-5 bullet points, 'standard' for 1-page "
            "executive summary, 'detailed' for section-by-section breakdown). "
            "Returns a JSON object with: title, author, summary_text, "
            "key_points (list of strings), and confidence_score. Use when the "
            "user asks 'what is this document about?', wants a summary, or needs "
            "to understand the main takeaways. Do NOT use for extracting specific "
            "data points like tables or numbers (use extract_structured_data) or "
            "for compliance checking (use audit_document_compliance)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "document_id": {
                    "type": "string",
                    "description": "Unique document identifier from the document store"
                },
                "summary_length": {
                    "type": "string",
                    "enum": ["brief", "standard", "detailed"],
                    "description": "Length of the summary to generate"
                }
            },
            "required": ["document_id", "summary_length"]
        }
    },
    {
        "name": "extract_structured_data",
        "description": (
            "Extracts structured data elements from a document: tables, named "
            "entities, dates, monetary amounts, and key-value pairs. Requires "
            "'document_id' (string) and 'extraction_targets' (list of strings "
            "from: 'tables', 'entities', 'dates', 'amounts', 'key_value_pairs'). "
            "Returns a JSON object with one key per target, each containing a "
            "list of extracted items with their page number, confidence score, "
            "and surrounding context. Use when the user needs specific data "
            "points, wants to populate a spreadsheet, or asks 'what are the "
            "dollar amounts in this contract?'. Do NOT use for understanding "
            "what the document is about (use summarize_document_key_points) or "
            "for checking regulatory compliance (use audit_document_compliance)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "document_id": {
                    "type": "string",
                    "description": "Unique document identifier"
                },
                "extraction_targets": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["tables", "entities", "dates", "amounts", "key_value_pairs"]
                    },
                    "description": "Types of data to extract"
                }
            },
            "required": ["document_id", "extraction_targets"]
        }
    },
    {
        "name": "audit_document_compliance",
        "description": (
            "Checks a document against a specified compliance policy or "
            "regulatory framework. Identifies missing required sections, "
            "policy violations, and regulatory red flags. Requires 'document_id' "
            "(string) and 'policy_framework' (one of: 'sox', 'gdpr', 'hipaa', "
            "'internal_policy', 'sec_filing'). Returns a JSON object with: "
            "compliance_score (0-100), violations (list with severity, "
            "description, page_number, and recommended_fix), missing_sections "
            "(list of required sections not found), and overall_risk_level "
            "('low', 'medium', 'high', 'critical'). Use when the user asks "
            "whether a document meets specific regulations or internal policies. "
            "Do NOT use for general document understanding (use "
            "summarize_document_key_points) or data extraction (use "
            "extract_structured_data)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "document_id": {
                    "type": "string",
                    "description": "Unique document identifier"
                },
                "policy_framework": {
                    "type": "string",
                    "enum": ["sox", "gdpr", "hipaa", "internal_policy", "sec_filing"],
                    "description": "The compliance framework to check against"
                }
            },
            "required": ["document_id", "policy_framework"]
        }
    },
]

print("SOLUTION -- Three Purpose-Specific Tools:")
print("=" * 65)
for tool in solution_split:
    result = score_tool_description(tool)
    print(f"\n  {tool['name']}: Grade {result['grade']} ({result['percentage']}%)")
    wrapped = textwrap.fill(tool['description'], width=58,
                            initial_indent="  Desc: ",
                            subsequent_indent="        ")
    print(wrapped)
    print()

# Routing test
print("\n\nRouting Test with Split Tools:")
print("=" * 65)
test_queries = [
    "What are the main findings in the Q4 earnings report?",
    "Pull out all the revenue figures from the contract.",
    "Does this document comply with GDPR requirements?",
    "Give me a brief summary of the merger proposal.",
    "Extract the table on page 5 with the quarterly numbers.",
]

for query in test_queries:
    result = mock_claude_improved_selection(query, solution_split)
    print(f"\n  \"{query[:55]}\"")
    print(f"  -> {result['selected_tool']} (confidence: {result['confidence']})")

### The Split Pattern

| Before (overlapping) | After (purpose-specific) |
|---|---|
| `analyze_content` | `summarize_document_key_points` |
| `analyze_document` | `extract_structured_data` |
| *(implicit)* | `audit_document_compliance` |

**Naming convention**: `[verb]_[object]_[qualifier]`
- The verb tells Claude the **action type** (summarize vs extract vs audit)
- The object tells Claude the **target** (document, data, compliance)
- The qualifier disambiguates further if needed (key_points, structured)

---

In [ ]:
#@title 🎧 Listen: Keyword Trap
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_keyword_trap.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 7: System Prompt Keyword Trap

Here is a subtle bug that even experienced developers miss. Keywords in your **system prompt** can create unintended associations that bias tool selection.

### The mechanism

When you write a system prompt like:

> *"You are an expert billing support agent. Always help with billing. Billing is our top priority. Resolve billing issues fast."*

The word "billing" appears so many times that it creates a strong contextual association. The model then over-indexes on tools with "billing" in their name or description -- even when the user asks about something completely unrelated like shipping.

Let's demonstrate this.

In [ ]:
# ============================================================
# 7A: Demonstrate the keyword trap
# ============================================================

# Scenario: You have a customer support agent with a system prompt
# that mentions "billing" extensively, and a tool called "get_billing_info".
# Even when the user asks about something unrelated, the system prompt
# keywords can push the model toward billing-related tools.

system_prompt_bad = """
You are a helpful customer support agent for BillingPro, a billing
and invoicing platform. You help customers with their billing questions,
billing disputes, billing cycles, and billing account management.
Our billing system is the best in the industry for billing automation.
Always prioritize billing-related requests and ensure billing accuracy.
"""

tools_for_demo = [
    {
        "name": "get_billing_info",
        "description": "Retrieves billing information for a customer account.",
        "input_schema": {"type": "object", "properties": {"account_id": {"type": "string"}}}
    },
    {
        "name": "get_shipping_status",
        "description": "Checks the shipping status of an order.",
        "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}}
    },
    {
        "name": "reset_password",
        "description": "Resets the password for a customer account.",
        "input_schema": {"type": "object", "properties": {"email": {"type": "string"}}}
    },
]

# Count how many times "billing" appears in the system prompt
billing_count = system_prompt_bad.lower().count("billing")
print("THE KEYWORD TRAP")
print("=" * 60)
print(f"\nSystem prompt mentions 'billing': {billing_count} times!")
print()

# Show how keyword saturation in the system prompt biases selection
def mock_selection_with_system_prompt(
    system_prompt: str, user_message: str, tools: list[dict]
) -> dict:
    """Simulates how system prompt keywords bias tool selection."""
    # Count keyword frequencies in system prompt
    sys_words = system_prompt.lower().split()
    sys_word_freq = Counter(sys_words)

    msg_lower = user_message.lower()
    scores = {}

    for tool in tools:
        desc_lower = tool["description"].lower()
        name_lower = tool["name"].lower()
        score = 0

        # Normal description matching
        for word in msg_lower.split():
            if len(word) > 3 and word in desc_lower:
                score += 1

        # BIAS: System prompt keyword amplification
        # When a keyword appears many times in the system prompt AND
        # in a tool name/description, it gets amplified
        for word in set(name_lower.replace("_", " ").split()) | set(desc_lower.split()):
            if word in sys_word_freq and sys_word_freq[word] >= 3:
                score += sys_word_freq[word] * 0.5  # Amplification effect

        scores[tool["name"]] = round(score, 1)

    selected = max(scores, key=scores.get)
    return {"selected_tool": selected, "scores": scores}

# Test: user asks about shipping (nothing to do with billing)
user_msg = "Where is my package? I ordered it last week."

result = mock_selection_with_system_prompt(system_prompt_bad, user_msg, tools_for_demo)

print(f"Customer: \"{user_msg}\"")
print(f"\nExpected tool: get_shipping_status")
print(f"Selected tool: {result['selected_tool']}")
print(f"\nScores: {result['scores']}")

if result["selected_tool"] != "get_shipping_status":
    print("\n*** MISROUTED! The 'billing' keyword saturation in the system prompt")
    print("    biased the model toward get_billing_info even though the user")
    print("    asked about shipping. ***")

In [ ]:
# ============================================================
# 7B: The fix -- Clean system prompt + explicit tool boundaries
# ============================================================

system_prompt_fixed = """
You are a helpful customer support agent for BillingPro, an invoicing
platform. You help customers with account management, order tracking,
and technical support. Route each request to the most specific tool
available. When in doubt, ask the customer to clarify.
"""

# Also improve the tool descriptions to be more self-contained
tools_fixed = [
    {
        "name": "get_billing_info",
        "description": (
            "Retrieves billing and payment information for a customer account, "
            "including current balance, payment history, and invoice details. "
            "Requires 'account_id' (string). Returns billing_summary, "
            "recent_invoices, and payment_method_on_file. Use when the customer "
            "asks about charges, invoices, payment history, or account balance. "
            "Do NOT use for shipping/delivery questions (use get_shipping_status) "
            "or account access issues (use reset_password)."
        ),
        "input_schema": {"type": "object", "properties": {"account_id": {"type": "string"}}}
    },
    {
        "name": "get_shipping_status",
        "description": (
            "Checks the current shipping and delivery status of a customer order. "
            "Returns carrier, tracking number, estimated delivery date, and status "
            "events. Requires 'order_id' (string). Use when the customer asks "
            "where their order/package is, when it will arrive, or about delivery "
            "issues. Do NOT use for billing questions (use get_billing_info) or "
            "account access (use reset_password)."
        ),
        "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}}
    },
    {
        "name": "reset_password",
        "description": (
            "Initiates a password reset flow for a customer account by sending "
            "a reset link to their registered email. Requires 'email' (valid "
            "email string). Returns confirmation with expiry time for the reset "
            "link. Use when the customer cannot log in, forgot their password, "
            "or needs account access restored. Do NOT use for billing (use "
            "get_billing_info) or order tracking (use get_shipping_status)."
        ),
        "input_schema": {"type": "object", "properties": {"email": {"type": "string"}}}
    },
]

billing_count_fixed = system_prompt_fixed.lower().count("billing")
print("AFTER THE FIX:")
print("=" * 60)
print(f"\nSystem prompt mentions 'billing': {billing_count_fixed} times (was {billing_count})")

# Re-test
result_fixed = mock_selection_with_system_prompt(
    system_prompt_fixed, user_msg, tools_fixed
)

print(f"\nCustomer: \"{user_msg}\"")
print(f"Expected tool: get_shipping_status")
print(f"Selected tool: {result_fixed['selected_tool']}")
print(f"Scores: {result_fixed['scores']}")

if result_fixed["selected_tool"] == "get_shipping_status":
    print("\n*** CORRECT! Cleaning the system prompt and adding explicit boundaries")
    print("    fixed the misrouting. ***")

In [ ]:
# ============================================================
# 7C: Visualize the bias effect
# ============================================================

test_cases = [
    ("Where is my package?", "get_shipping_status"),
    ("I forgot my password", "reset_password"),
    ("What is my current balance?", "get_billing_info"),
    ("When will my order arrive?", "get_shipping_status"),
    ("I can't log into my account", "reset_password"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (sys_prompt, tools_set, title) in zip(axes, [
    (system_prompt_bad, tools_for_demo, "Before Fix\n(keyword-saturated prompt)"),
    (system_prompt_fixed, tools_fixed, "After Fix\n(clean prompt + boundaries)")
]):
    correct = 0
    total = len(test_cases)

    for msg, expected in test_cases:
        result = mock_selection_with_system_prompt(sys_prompt, msg, tools_set)
        if result["selected_tool"] == expected:
            correct += 1

    accuracy = correct / total * 100
    wrong = total - correct

    bars = ax.bar(["Correct", "Wrong"], [correct, wrong],
                  color=["#51cf66", "#ff6b6b"], alpha=0.85, edgecolor="white",
                  linewidth=2)
    ax.set_ylim(0, total + 1)
    ax.set_title(f"{title}\nAccuracy: {accuracy:.0f}%", fontsize=12, fontweight="bold")
    ax.set_ylabel("Number of test cases", fontsize=11)

    for bar, count in zip(bars, [correct, wrong]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                str(count), ha='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### The Keyword Trap -- Rules to Remember

1. **Do not repeat domain keywords excessively** in your system prompt. Say it once clearly.
2. **Tool descriptions must be self-contained.** They should route correctly even if you swapped the system prompt.
3. **Negative boundaries are your defense.** "Do NOT use for X" prevents keyword bleed from the system prompt.
4. **Test with adversarial phrasings.** If your system prompt says "billing" 8 times, test with a non-billing request.

---

In [ ]:
#@title 🎧 Listen: Linter Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_linter_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 8: Mini-Project -- Build a Tool Description Linter

Now let's build a practical tool: a **linter** that automatically flags common anti-patterns in tool descriptions. You could integrate this into your CI/CD pipeline to catch bad descriptions before deployment.

The linter checks 9 rules:

| Rule | Severity | What it catches |
|---|---|---|
| `TOO_SHORT` | ERROR | Description under 10 words |
| `NO_ACTION_VERB` | WARNING | Missing clear action verb at start |
| `NO_NEGATIVE_BOUNDARY` | ERROR | No "Do NOT use" clause |
| `NO_RETURN_DESCRIPTION` | WARNING | No description of return value |
| `GENERIC_NAME` | WARNING | Tool name like `handle_data` |
| `VAGUE_LANGUAGE` | WARNING | Words like "stuff", "things", "various" |
| `NO_INPUT_CONSTRAINTS` | WARNING | Parameters lack descriptions/enums |
| `DESCRIPTION_MIRRORS_NAME` | ERROR | Description just restates the name |
| `NO_EXAMPLES` | INFO | No concrete examples (e.g., ...) |

In [ ]:
# ============================================================
# 8A: Define the anti-pattern rules
# ============================================================

def lint_tool_description(tool: dict) -> list[dict]:
    """
    Lints a tool definition for common description anti-patterns.
    Returns a list of findings with severity and recommendation.
    """
    findings = []
    desc = tool.get("description", "")
    name = tool.get("name", "")
    schema = tool.get("input_schema", {})
    props = schema.get("properties", {})
    desc_lower = desc.lower()
    word_count = len(desc.split())

    # Rule 1: Too short
    if word_count < 10:
        findings.append({
            "rule": "TOO_SHORT",
            "severity": "ERROR",
            "message": f"Description is only {word_count} words. Minimum recommended: 30.",
            "fix": "Add: what it does, expected inputs, return value, when to use, when NOT to use."
        })

    # Rule 2: No action verb
    action_verbs = [
        "retrieves", "fetches", "creates", "updates", "deletes", "calculates",
        "generates", "sends", "searches", "modifies", "validates", "converts",
        "exports", "imports", "analyzes", "checks", "initiates", "extracts",
        "changes"
    ]
    if not any(v in desc_lower for v in action_verbs):
        findings.append({
            "rule": "NO_ACTION_VERB",
            "severity": "WARNING",
            "message": "Description does not start with a clear action verb.",
            "fix": f"Begin with a verb like: {', '.join(action_verbs[:6])}..."
        })

    # Rule 3: Missing negative boundary
    if "do not" not in desc_lower and "don't" not in desc_lower and "not for" not in desc_lower:
        findings.append({
            "rule": "NO_NEGATIVE_BOUNDARY",
            "severity": "ERROR",
            "message": "No 'Do NOT use' clause found. This is the #1 cause of tool overlap.",
            "fix": "Add: 'Do NOT use for X (use <other_tool> instead).'"
        })

    # Rule 4: No return value description
    if "returns" not in desc_lower and "return" not in desc_lower:
        findings.append({
            "rule": "NO_RETURN_DESCRIPTION",
            "severity": "WARNING",
            "message": "No description of what the tool returns.",
            "fix": "Add: 'Returns a JSON object with: field1, field2, ...'"
        })

    # Rule 5: Generic tool name
    generic_stems = ["handle", "process", "do", "run", "manage", "get", "set",
                     "execute", "perform"]
    name_parts = name.lower().split("_")
    if name_parts[0] in generic_stems and len(name_parts) <= 2:
        findings.append({
            "rule": "GENERIC_NAME",
            "severity": "WARNING",
            "message": f"Tool name '{name}' is too generic. Ambiguous for routing.",
            "fix": "Use format: [specific_verb]_[object]_[qualifier], e.g., 'fetch_stock_quote'"
        })

    # Rule 6: Vague language
    vague_words = ["stuff", "things", "data", "info", "various", "general",
                   "miscellaneous", "other", "etc"]
    found_vague = [w for w in vague_words if w in desc_lower.split()]
    if found_vague:
        findings.append({
            "rule": "VAGUE_LANGUAGE",
            "severity": "WARNING",
            "message": f"Vague words detected: {found_vague}",
            "fix": "Replace with specific terms. 'data' -> 'stock price quote', 'info' -> 'billing summary'"
        })

    # Rule 7: No input constraints
    has_enum = any(isinstance(p, dict) and "enum" in p for p in props.values())
    has_desc = any(isinstance(p, dict) and "description" in p for p in props.values())
    has_required = "required" in schema
    if not has_enum and not has_desc and not has_required and len(props) > 0:
        findings.append({
            "rule": "NO_INPUT_CONSTRAINTS",
            "severity": "WARNING",
            "message": "Parameters have no descriptions, enums, or required markers.",
            "fix": "Add 'description' and 'enum' to each property. Mark required fields."
        })

    # Rule 8: Description too similar to name
    name_words = set(name.lower().replace("_", " ").split())
    stop_words = {"a", "the", "and", "or", "for", "in", "to", "of", "is", "it"}
    desc_words = set(desc_lower.split()) - stop_words
    unique_desc_words = desc_words - name_words
    if len(unique_desc_words) < 5:
        findings.append({
            "rule": "DESCRIPTION_MIRRORS_NAME",
            "severity": "ERROR",
            "message": "Description merely restates the tool name with no additional detail.",
            "fix": "Description should explain HOW, WHEN, WHAT FORMAT -- not just repeat the name."
        })

    # Rule 9: No concrete examples
    if "e.g." not in desc_lower and "for example" not in desc_lower and "'" not in desc:
        findings.append({
            "rule": "NO_EXAMPLES",
            "severity": "INFO",
            "message": "No concrete examples found in description.",
            "fix": "Add examples: \"e.g., 'AAPL'\" or \"for example, 'cancel'\""
        })

    return findings


# Test the linter on our bad tools
print("LINTING BAD TOOLS:")
print("=" * 65)
for tool in bad_tools:
    findings = lint_tool_description(tool)
    print(f"\n  {tool['name']}  ({len(findings)} issue{'s' if len(findings) != 1 else ''}):")
    if not findings:
        print("    No issues found.")
    for f in findings:
        icon = {"ERROR": "[!]", "WARNING": "[~]", "INFO": "[i]"}[f["severity"]]
        print(f"    {icon} {f['severity']:7s} | {f['rule']}")
        print(f"        {f['message']}")
        print(f"        Fix: {f['fix']}")

In [ ]:
# ============================================================
# 8B: Lint the good tools (should pass cleanly)
# ============================================================

print("LINTING GOOD TOOLS:")
print("=" * 65)
for tool in good_tools:
    findings = lint_tool_description(tool)
    error_count = sum(1 for f in findings if f["severity"] == "ERROR")
    warn_count = sum(1 for f in findings if f["severity"] == "WARNING")
    info_count = sum(1 for f in findings if f["severity"] == "INFO")

    status = "PASS" if error_count == 0 else "FAIL"
    print(f"\n  [{status}] {tool['name']}")
    if not findings:
        print("    All checks passed.")
    for f in findings:
        icon = {"ERROR": "[!]", "WARNING": "[~]", "INFO": "[i]"}[f["severity"]]
        print(f"    {icon} {f['severity']:7s} | {f['rule']}: {f['message']}")

In [ ]:
# ============================================================
# 8C: Lint the solution split tools
# ============================================================

print("LINTING SOLUTION SPLIT TOOLS:")
print("=" * 65)
for tool in solution_split:
    findings = lint_tool_description(tool)
    error_count = sum(1 for f in findings if f["severity"] == "ERROR")

    status = "PASS" if error_count == 0 else "FAIL"
    print(f"\n  [{status}] {tool['name']}")
    if not findings:
        print("    All checks passed.")
    for f in findings:
        icon = {"ERROR": "[!]", "WARNING": "[~]", "INFO": "[i]"}[f["severity"]]
        print(f"    {icon} {f['severity']:7s} | {f['rule']}: {f['message']}")

In [ ]:
# ============================================================
# 8D: Visualize lint results across all tool sets
# ============================================================

def count_findings_by_severity(tools: list[dict]) -> dict:
    """Count lint findings by severity across a set of tools."""
    counts = {"ERROR": 0, "WARNING": 0, "INFO": 0}
    for tool in tools:
        for finding in lint_tool_description(tool):
            counts[finding["severity"]] += 1
    return counts

all_tool_sets = {
    "Bad Tools\n(Section 2)": bad_tools,
    "Good Tools\n(Section 3)": good_tools,
    "Overlapping\n(Section 6 before)": overlapping_tools,
    "Split Tools\n(Section 6 after)": solution_split,
}

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(all_tool_sets))
width = 0.25

severity_colors = {"ERROR": "#ff6b6b", "WARNING": "#ffd43b", "INFO": "#74c0fc"}

for i, (severity, color) in enumerate(severity_colors.items()):
    counts = [count_findings_by_severity(tools)[severity]
              for tools in all_tool_sets.values()]
    bars = ax.bar(x + i * width, counts, width, label=severity,
                  color=color, edgecolor="white", linewidth=1.5)
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    str(count), ha='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Tool Set', fontsize=12)
ax.set_ylabel('Number of Findings', fontsize=12)
ax.set_title('Lint Findings by Severity Across Tool Sets', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(all_tool_sets.keys(), fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, max(
    max(count_findings_by_severity(tools).values())
    for tools in all_tool_sets.values()
) + 2)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8E: Build the full linter report function (CI/CD ready)
# ============================================================

def lint_report(tools: list[dict], report_name: str = "Tool Lint Report") -> dict:
    """
    Generates a complete lint report for a set of tool definitions.
    This is the function you would integrate into CI/CD.
    Returns a summary dict with pass/fail status.
    """
    print(f"\n{'=' * 65}")
    print(f"  {report_name}")
    print(f"{'=' * 65}")

    total_errors = 0
    total_warnings = 0
    total_info = 0

    for tool in tools:
        findings = lint_tool_description(tool)
        errors = [f for f in findings if f["severity"] == "ERROR"]
        warnings = [f for f in findings if f["severity"] == "WARNING"]
        infos = [f for f in findings if f["severity"] == "INFO"]

        total_errors += len(errors)
        total_warnings += len(warnings)
        total_info += len(infos)

        status = "FAIL" if errors else "WARN" if warnings else "PASS"
        print(f"\n  [{status}] {tool['name']}")

        for f in findings:
            icon = {"ERROR": "[!]", "WARNING": "[~]", "INFO": "[i]"}[f["severity"]]
            print(f"       {icon} {f['rule']}: {f['message']}")

    print(f"\n{'=' * 65}")
    print(f"  Summary: {len(tools)} tools checked")
    print(f"    Errors:   {total_errors}")
    print(f"    Warnings: {total_warnings}")
    print(f"    Info:     {total_info}")

    passed = total_errors == 0
    if not passed:
        print(f"\n  RESULT: FAILED -- {total_errors} error(s) must be fixed before deploy.")
    elif total_warnings > 0:
        print(f"\n  RESULT: PASSED WITH WARNINGS -- review {total_warnings} warning(s).")
    else:
        print(f"\n  RESULT: PASSED -- all tools meet quality standards.")
    print(f"{'=' * 65}")

    return {
        "passed": passed,
        "errors": total_errors,
        "warnings": total_warnings,
        "info": total_info,
        "tools_checked": len(tools)
    }

# Run the full report on both sets
result_bad = lint_report(bad_tools, "Audit: Original (Bad) Tools")
result_good = lint_report(good_tools, "Audit: Improved (Good) Tools")

print("\n\nCI/CD Integration Summary:")
print(f"  Bad tools  -> {'BLOCKED' if not result_bad['passed'] else 'DEPLOYED'}")
print(f"  Good tools -> {'BLOCKED' if not result_good['passed'] else 'DEPLOYED'}")

### Integrating the Linter

In a production setup, you would:

1. **Store tool definitions as JSON files** in your repository
2. **Run `lint_report()` in CI** (e.g., GitHub Actions) on every PR that modifies tool definitions
3. **Fail the build** if any `ERROR`-level finding is present
4. **Require review** for `WARNING`-level findings

In [ ]:
# Example: GitHub Actions integration (pseudocode)
#
# import sys
# tools = load_tools_from_json("tools/*.json")
# result = lint_report(tools, "PR Tool Audit")
# if not result["passed"]:
#     sys.exit(1)  # Fail the CI build

This catches description quality issues *before* they reach production and cause misrouting.

---

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 9: Key Takeaways + What's Next

### What You Learned

| # | Concept | Key Insight |
|---|---|---|
| 1 | **Vague descriptions break routing** | The same model with better descriptions produces reliable tool selection |
| 2 | **5-component anatomy** | What it does, inputs, outputs, when to use, when NOT to use |
| 3 | **Negative boundaries are critical** | "Do NOT use for X" is the single most impactful addition |
| 4 | **Overlapping tools must be split** | Use `[verb]_[object]_[qualifier]` naming to create natural separation |
| 5 | **System prompt keywords create bias** | Keyword saturation bleeds into tool selection via contextual association |
| 6 | **Automate quality checks** | A linter catches anti-patterns before deployment |

### Decision Framework

When designing tool descriptions, ask yourself:

1. **Could Claude confuse this tool with another?** If yes, add negative boundaries.
2. **Does the name alone suggest the scope?** If not, rename it with `[verb]_[object]_[qualifier]`.
3. **Are all valid input values documented?** If not, add enums and format strings.
4. **Would a new team member know when to use this tool?** If not, add use-case examples.
5. **Does the description work without the system prompt?** If not, make it self-contained.

### Exam Relevance

On the Claude Certified Architect exam, tool description questions typically appear as:
- "Which tool definition would most reliably route this user request?"
- "What is wrong with this tool description?" (identify the missing component)
- "Two tools overlap. How would you fix the routing?"

The patterns in this notebook cover all three question types.

### What's Next

In **Notebook 2: MCP Server Architecture**, we will build on these tool description skills and explore:
- How tools are exposed through the Model Context Protocol (MCP)
- Server-side tool registration and schema validation
- Client-server tool negotiation
- Building your first MCP server with proper tool definitions

---

*Claude Certified Architect Prep -- Pod 2: Tool Design & MCP Integration*

In [ ]:
# ============================================================
# Final: Verify all sections are present and review
# ============================================================

print("Notebook 1: Crafting Effective Tool Descriptions")
print("=" * 55)
print()
print("Section Checklist:")
sections = [
    "1. Title + Setup",
    "2. Warm-Up: The Problem (misrouting demo)",
    "3. Core Concept: Tool Description Anatomy",
    "4. Guided Exercise: Rewrite Descriptions",
    "5. Visual: Description Quality Scorer",
    "6. Exercise: Rename and Split Overlapping Tools",
    "7. System Prompt Keyword Trap",
    "8. Mini-Project: Tool Description Linter",
    "9. Key Takeaways + What's Next",
]
for s in sections:
    print(f"  [x] {s}")

print()
print("Key functions defined:")
functions = [
    "mock_claude_tool_selection()        -- basic keyword-overlap routing",
    "mock_claude_improved_selection()    -- semantic routing with boundaries",
    "mock_selection_with_system_prompt() -- system prompt bias simulation",
    "score_tool_description()            -- 5-criteria quality scorer (0-50)",
    "lint_tool_description()             -- anti-pattern linter (9 rules)",
    "lint_report()                       -- CI/CD-ready lint report generator",
    "plot_radar_comparison()             -- radar chart: before vs after",
    "plot_individual_scores()            -- bar chart: per-tool breakdown",
]
for f in functions:
    print(f"  - {f}")

print()
print("All code is self-contained and runnable without an API key.")
print("Proceed to Notebook 2: MCP Server Architecture when ready.")